# Baseline model — DummyClassifier

**IT2-14 / IT2-15** · Binary sentiment classification (0 = negative, 1 = positive).

IT2-14 establishes three `DummyClassifier` baselines using the sequence **build → fit → predict train → predict test → evaluate**. IT2-15 saves the comparison metrics to CSV and exports the test confusion matrix for the `most_frequent` reference baseline.


## 1. Load the prepared train and test splits

We use the existing stratified splits so every model is evaluated on the same observations. The dummy classifier does not use text content to learn sentiment; the feature input is only a placeholder.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier

root = Path.cwd().resolve()
if not (root / "src" / "evaluate.py").exists():
    root = root.parent
if not (root / "src" / "evaluate.py").exists():
    raise FileNotFoundError("Run this notebook from the project root or notebooks/ directory")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.evaluate import evaluate_train_test

SEED = 1234
train = pd.read_csv(root / "data" / "processed" / "train.csv")
test = pd.read_csv(root / "data" / "processed" / "test.csv")

required_columns = {"comment_id", "text", "clean_text", "label"}
assert required_columns.issubset(train.columns)
assert required_columns.issubset(test.columns)
assert len(train) == 800 and len(test) == 200
assert train["label"].value_counts().to_dict() == {0: 400, 1: 400}
assert test["label"].value_counts().to_dict() == {0: 100, 1: 100}

print("Train:", train.shape, "Test:", test.shape)
print("Train labels:", train["label"].value_counts().sort_index().to_dict())
print("Test labels:", test["label"].value_counts().sort_index().to_dict())

## 2. Prepare labels and placeholder features

`DummyClassifier` ignores feature values when making its baseline predictions. We supply a constant feature column, and keep the true labels separate.

In [ ]:
X_train = np.zeros((len(train), 1))
X_test = np.zeros((len(test), 1))
y_train = train["label"].to_numpy()
y_test = test["label"].to_numpy()

## 3. Build, fit, predict, and evaluate

- **most_frequent:** always predicts the most frequent training class (in a tie, scikit-learn chooses one class).
- **stratified:** samples labels according to the training class distribution.
- **uniform:** samples classes with equal probability.

We fix the random seed so the random baselines are reproducible.

In [ ]:
strategies = ("most_frequent", "stratified", "uniform")
results = []

for strategy in strategies:
    # Build
    model = DummyClassifier(strategy=strategy, random_state=SEED)
    # Fit
    model.fit(X_train, y_train)
    # Predict train
    pred_train = model.predict(X_train)
    # Predict test
    pred_test = model.predict(X_test)
    # Evaluate both splits
    metrics = evaluate_train_test(
        name=f"dummy_{strategy}",
        y_train=y_train,
        p_train=pred_train,
        y_test=y_test,
        p_test=pred_test,
    )
    results.append(metrics)

baseline_results = pd.concat(results, ignore_index=True)
display(baseline_results.round(4))

## 4. Inspect the main baseline metrics

In [ ]:
display(
    baseline_results[
        ["model", "split", "accuracy", "precision", "recall", "f1", "f1_macro", "kappa"]
    ].round(4)
)

reference = baseline_results[
    (baseline_results["model"] == "dummy_most_frequent")
    & (baseline_results["split"] == "test")
].iloc[0]
assert np.isclose(reference["accuracy"], 0.5)
assert np.isclose(reference["kappa"], 0.0)
print("most_frequent test accuracy:", reference["accuracy"])
print("most_frequent test kappa:", reference["kappa"])

## 5. Interpretation

The dataset is balanced: half the examples are positive and half are negative in both splits. Therefore, a model that always predicts one class (`most_frequent`) correctly classifies exactly half the examples, giving **accuracy = 0.50**. Its **Cohen's kappa = 0** because its agreement is no better than expected from the class frequencies.

`stratified` and `uniform` make random predictions, so their exact scores depend on the seed and sample size. Their accuracy is expected to be near 0.50 and kappa near 0, not necessarily exactly those values. These models do not learn sentiment from the comments.

**IT2-15:** export the complete train/test comparison and the `most_frequent` test confusion matrix below. Compare later trained models against this chance-level reference.


## 6. Save baseline metrics (IT2-15)

Export the six train/test rows for the three baseline strategies using the existing project helper. The output is `outputs/metrics/baseline_metrics.csv`.


In [ ]:
from src.evaluate import save_metrics

metrics_path = root / "outputs" / "metrics" / "baseline_metrics.csv"
metrics_path.parent.mkdir(parents=True, exist_ok=True)
save_metrics(baseline_results, "baseline")
assert metrics_path.is_file(), f"Expected metrics file was not created: {metrics_path}"
saved_metrics = pd.read_csv(metrics_path)
assert len(saved_metrics) == 6
assert set(saved_metrics["model"]) == {f"dummy_{strategy}" for strategy in strategies}
assert set(saved_metrics["split"]) == {"train", "test"}
print("Saved metrics:", metrics_path)
display(saved_metrics.round(4))


## 7. Export test confusion matrix (IT2-15)

The confusion matrix uses the `most_frequent` model on the test split. Rows are actual labels, columns are predicted labels, ordered as negative (0) and positive (1).


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

reference_model = DummyClassifier(strategy="most_frequent", random_state=SEED)
reference_model.fit(X_train, y_train)
reference_pred_test = reference_model.predict(X_test)
cm = confusion_matrix(y_test, reference_pred_test, labels=[0, 1])
assert cm.shape == (2, 2)
assert cm.sum() == len(y_test)

figure_path = root / "outputs" / "figures" / "baseline_confusion_matrix.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative (0)", "Positive (1)"],
).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title("Dummy most_frequent — test confusion matrix")
fig.tight_layout()
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.close(fig)
assert figure_path.is_file() and figure_path.stat().st_size > 0
print("Test confusion matrix:\n", cm)
print("Saved figure:", figure_path)


## 8. Chance-level comparison

On this balanced binary test set, the `most_frequent` baseline has accuracy 0.50 and Cohen's kappa 0.00. A trained model should exceed this reference, preferably on both accuracy and kappa, to demonstrate predictive value beyond the trivial majority-class decision. The stochastic strategies may be slightly above or below 0.50 on this finite sample, so their measured scores should be reported rather than assumed.
